# B0 — BIRD Data Loading & Preprocessing

Loads the raw BIRD dataset (51 museum visitors, Nancy Museum of Fine Arts) and produces two merged CSVs for the RML mapping pipeline:
- `bird/files/csv/bird_trajectories.csv` — all trajectory positions with observation linkage
- `bird/files/csv/artworks.csv` — artwork metadata with category mapping

In [ ]:
import pandas as pd
import os
from pathlib import Path

import sys
sys.path.insert(0, os.path.abspath('helpers'))
from artwork_categories import resolve_category

In [ ]:
# Paths
BIRD_RAW = Path('../../data/bird/bird-dataset-main/data')
OUTPUT_DIR = Path('../files/csv')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data: {BIRD_RAW.resolve()}')
print(f'Output:   {OUTPUT_DIR.resolve()}')

## 1. Load Normalized Trajectories

In [ ]:
traj_dir = BIRD_RAW / 'normalized_trajectories'
traj_files = sorted(traj_dir.glob('*_traj_normalized.csv'))
print(f'Found {len(traj_files)} trajectory files')

all_trajectories = []
for f in traj_files:
    traj_id = f.stem.replace('_traj_normalized', '')
    df = pd.read_csv(f)
    df['traj_id'] = traj_id
    df['id'] = range(len(df))  # position index within trajectory
    all_trajectories.append(df)

trajectories = pd.concat(all_trajectories, ignore_index=True)
trajectories.rename(columns={'floorNumber': 'floor'}, inplace=True)
print(f'Total positions: {len(trajectories)}')
print(f'Trajectories: {trajectories["traj_id"].nunique()}')
trajectories.head()

## 2. Load Artwork Observations (Start/End)

In [ ]:
start_dir = BIRD_RAW / 'start_obs_artworks'
end_dir = BIRD_RAW / 'end_obs_artworks'

all_observations = []

for start_file in sorted(start_dir.glob('items_*.csv')):
    traj_id = start_file.stem.replace('items_', '')
    end_file = end_dir / f'items_{traj_id}_end.csv'

    if not end_file.exists():
        print(f'Warning: no end file for {traj_id}')
        continue

    starts = pd.read_csv(start_file).sort_values('timestamp')
    ends = pd.read_csv(end_file).sort_values('timestamp')

    # Group by paintingId and pair nth-start with nth-end within each group
    start_groups = starts.groupby('paintingId')
    end_groups = ends.groupby('paintingId')

    for painting_id, s_group in start_groups:
        if painting_id not in end_groups.groups:
            continue
        e_group = end_groups.get_group(painting_id)

        n_pairs = min(len(s_group), len(e_group))
        for s, e in zip(s_group.iloc[:n_pairs].itertuples(), e_group.iloc[:n_pairs].itertuples()):
            dwell_time = e.timestamp - s.timestamp
            if pd.isna(dwell_time) or dwell_time < 0:
                continue
            all_observations.append({
                'traj_id': traj_id,
                'artwork_id': painting_id,
                'start_time': s.timestamp,
                'end_time': e.timestamp,
                'dwell_time': round(dwell_time),
                'floor': s.floorNumber,
            })

# Sort by trajectory and start time for consistent ordering
all_observations.sort(key=lambda x: (x['traj_id'], x['start_time']))
observations = pd.DataFrame(all_observations)
observations['obs_id'] = range(len(observations))
print(f'Total observations: {len(observations)}')
print(f'Unique artworks observed: {observations["artwork_id"].nunique()}')
print(f'Mean dwell time: {observations["dwell_time"].mean():.1f}s')

## 3. Link Trajectory Positions to Observations

For each observation window (start_time → end_time), mark trajectory positions that fall within it as stops linked to that artwork.

In [ ]:
# Initialize columns
trajectories['is_stop'] = False
trajectories['obs_id'] = ''
trajectories['artwork_id'] = ''
trajectories['dwell_time'] = 0.0

for traj_id in trajectories['traj_id'].unique():
    traj_mask = trajectories['traj_id'] == traj_id
    traj_obs = observations[observations['traj_id'] == traj_id]

    for _, obs in traj_obs.iterrows():
        # Find positions within the observation time window
        time_mask = (
            traj_mask &
            (trajectories['timestamp'] >= obs['start_time']) &
            (trajectories['timestamp'] <= obs['end_time'])
        )

        if time_mask.any():
            trajectories.loc[time_mask, 'is_stop'] = True
            trajectories.loc[time_mask, 'obs_id'] = str(obs['obs_id'])
            trajectories.loc[time_mask, 'artwork_id'] = obs['artwork_id']
            trajectories.loc[time_mask, 'dwell_time'] = obs['dwell_time']

stop_count = trajectories['is_stop'].sum()
total = len(trajectories)
print(f'Positions marked as stops: {stop_count} / {total} ({100*stop_count/total:.1f}%)')

## 4. Add Username Column

Each trajectory ID becomes a unique visitor identifier.

In [ ]:
trajectories['username'] = 'visitor_' + trajectories['traj_id']
print(f'Unique visitors: {trajectories["username"].nunique()}')

## 5. Load & Process Artworks Dataset

In [ ]:
artworks_raw = pd.read_csv(BIRD_RAW / 'artworks_dataset.csv')
print(f'Raw artworks: {len(artworks_raw)}')
print(f'Columns: {list(artworks_raw.columns)}')

In [ ]:
# Extract and rename relevant columns
artworks = artworks_raw[['image id', 'painting name', 'artist', 'date', 'Type',
                          'floor', 'wall_id', 'room_id']].copy()
artworks.columns = ['image_id', 'painting_name', 'artist', 'date', 'type',
                     'floor', 'wall_id', 'room_id']

# Map raw type to category
artworks['category'] = artworks['type'].apply(resolve_category)

# Clean up missing values
artworks['painting_name'] = artworks['painting_name'].fillna('Unknown')
artworks['artist'] = artworks['artist'].fillna('Unknown')
artworks['date'] = artworks['date'].fillna('Unknown')
artworks['type'] = artworks['type'].fillna('-')

print(f'Processed artworks: {len(artworks)}')
print('\nCategory distribution:')
print(artworks['category'].value_counts())
print('\nType distribution (top 10):')
print(artworks['type'].value_counts().head(10))

## 6. Validate Artwork Linkage

In [ ]:
# Check that observed artwork IDs exist in the artworks dataset
observed_ids = set(observations['artwork_id'].unique())
artwork_ids = set(artworks['image_id'].values)
missing = observed_ids - artwork_ids

if missing:
    print(f'Warning: {len(missing)} observed artwork IDs not in artworks dataset:')
    for m in sorted(missing)[:10]:
        print(f'  {m}')
else:
    print(f'All {len(observed_ids)} observed artwork IDs found in artworks dataset')

## 7. Summary Statistics

In [ ]:
print('=== BIRD Dataset Summary ===')
print(f'Visitors/Trajectories: {trajectories["traj_id"].nunique()}')
print(f'Total positions: {len(trajectories)}')
print(f'Total artworks: {len(artworks)}')
print(f'Total observations: {len(observations)}')
print(f'Unique artworks observed: {observations["artwork_id"].nunique()}')
print('\nDwell time statistics:')
print(f'  Mean:   {observations["dwell_time"].mean():.1f}s')
print(f'  Median: {observations["dwell_time"].median():.1f}s')
print(f'  Std:    {observations["dwell_time"].std():.1f}s')
print(f'  Min:    {observations["dwell_time"].min():.1f}s')
print(f'  Max:    {observations["dwell_time"].max():.1f}s')

print('\nObservations per trajectory:')
obs_per_traj = observations.groupby('traj_id').size()
print(f'  Mean:   {obs_per_traj.mean():.1f}')
print(f'  Min:    {obs_per_traj.min()}')
print(f'  Max:    {obs_per_traj.max()}')

print('\nFloor distribution (positions):')
print(trajectories['floor'].value_counts().sort_index())

## 8. Save Output CSVs

In [ ]:
# Save trajectory CSV
traj_output = trajectories[['traj_id', 'id', 'timestamp', 'x', 'y', 'floor',
                             'is_stop', 'obs_id', 'artwork_id', 'dwell_time', 'username']]
traj_path = OUTPUT_DIR / 'bird_trajectories.csv'
traj_output.to_csv(traj_path, index=False)
print(f'Saved {len(traj_output)} trajectory positions to {traj_path}')

# Save artworks CSV
art_path = OUTPUT_DIR / 'artworks.csv'
artworks.to_csv(art_path, index=False)
print(f'Saved {len(artworks)} artworks to {art_path}')

# Save observations CSV (useful for enrichment notebook)
obs_path = OUTPUT_DIR / 'observations.csv'
observations.to_csv(obs_path, index=False)
print(f'Saved {len(observations)} observations to {obs_path}')

In [ ]:
# Verify output
verify = pd.read_csv(traj_path)
print('Verification — bird_trajectories.csv:')
print(f'  Columns: {list(verify.columns)}')
print(f'  Shape: {verify.shape}')
print(f'  Trajectories: {verify["traj_id"].nunique()}')
print(f'  Stops: {verify["is_stop"].sum()}')
verify.head()